# 02 — Fetch & View Portfolio Holdings

This notebook loads your saved access tokens and pulls investment holdings from all connected brokerages.
It gives you a unified view of your entire portfolio across Stash, Robinhood, SoFi, and Fidelity.

**Prerequisites:**
- Run notebook `01_connect_accounts.ipynb` first to generate `access_tokens.json`

## Setup & Load Tokens

In [1]:

import os
from dotenv import load_dotenv
load_dotenv(os.path.join("..", ".env"))

PLAID_CLIENT_ID = os.getenv("PLAID_CLIENT_ID")
PLAID_SECRET = os.getenv("PLAID_SECRET")

if not PLAID_CLIENT_ID or not PLAID_SECRET:
    print("⚠️  Plaid credentials missing in .env — skipping Plaid fetch.")
    print("   Please verify manual accounts or CSV imports instead.")
    # Gracefully exit the notebook execution
    
import json
import os
import sys

import pandas as pd

sys.path.insert(0, os.path.abspath(".."))

from plaid_config import get_plaid_client

client = get_plaid_client()

# Load saved access tokens
TOKENS_FILE = os.path.join("..", "access_tokens.json")

if not os.path.exists(TOKENS_FILE):
    raise FileNotFoundError(
        "access_tokens.json not found. Run notebook 01 first to connect accounts."
    )

with open(TOKENS_FILE, "r") as f:
    access_tokens = json.load(f)

print(f"✅ Loaded {len(access_tokens)} access tokens: {list(access_tokens.keys())}")


✅ Loaded 5 access tokens: ['Robinhood', 'Sofi', 'Stash', 'Acorns', 'Wealthfront']


## Pull Holdings from All Brokerages

In [2]:
from plaid.model.investments_holdings_get_request import InvestmentsHoldingsGetRequest

# ── Account category classifier ────────────────────────────────────────────
ACCOUNT_CATEGORY_MAP = {
    "brokerage": "taxable", "stock plan": "taxable",
    "ira": "retirement", "roth": "retirement", "roth 401k": "retirement",
    "401k": "retirement", "401a": "retirement", "403b": "retirement",
    "457b": "retirement", "sep ira": "retirement", "simple ira": "retirement",
    "pension": "retirement", "hsa": "retirement",
    "checking": "liquid", "savings": "liquid", "money market": "liquid",
    "cash management": "liquid", "cd": "liquid",
}

# ── Manual liquid accounts NOT connected to Plaid ─────────────────────────
# Update these whenever balances change significantly
MANUAL_LIQUID_ACCOUNTS = {
    "BofA Checking": 4614.00,
}

def classify_account(plaid_subtype):
    if plaid_subtype is None:
        return "other"
    return ACCOUNT_CATEGORY_MAP.get(str(plaid_subtype).lower(), "other")

# ── Fetch holdings from all brokerages ────────────────────────────────────
all_holdings = []
all_cash_accounts = []  # checking/savings balances from Plaid

for nickname, token in access_tokens.items():
    print(f"Fetching holdings from '{nickname}'...", end=" ")

    response = client.investments_holdings_get(
        InvestmentsHoldingsGetRequest(access_token=token)
    )

    # Build security lookup: security_id → security object
    securities = {s.security_id: s for s in response.securities}

    # Build account lookup: account_id → {name, type, subtype, mask}
    accounts = {
        a.account_id: {
            "name": a.name,
            "type": str(a.type) if a.type else None,
            "subtype": str(a.subtype) if a.subtype else None,
            "mask": a.mask,
        }
        for a in response.accounts
    }

    # ── Extract cash/liquid account balances ──────────────────────────────
    for acct in response.accounts:
        acct_subtype = str(acct.subtype) if acct.subtype else None
        category = classify_account(acct_subtype)
        if category == "liquid" and acct.balances and acct.balances.current is not None:
            all_cash_accounts.append({
                "brokerage": nickname,
                "account_name": acct.name,
                "account_subtype": acct_subtype,
                "category": "liquid",
                "balance": float(acct.balances.current),
            })

    for holding in response.holdings:
        sec = securities.get(holding.security_id)
        acct = accounts.get(holding.account_id, {})
        acct_subtype = acct.get("subtype")
        acct_category = classify_account(acct_subtype)

        all_holdings.append({
            "brokerage": nickname,
            "ticker": sec.ticker_symbol if sec else None,
            "name": sec.name if sec else None,
            "type": sec.type if sec else None,
            "quantity": holding.quantity,
            "cost_basis": holding.cost_basis,
            "price": holding.institution_price,
            "value": holding.institution_value,
            "currency": holding.iso_currency_code,
            # Account metadata
            "account_id": holding.account_id,
            "account_name": acct.get("name"),
            "account_subtype": acct_subtype,
            "account_type": acct_category,
        })

    # Show account breakdown for this brokerage
    acct_types = {}
    for a in accounts.values():
        sub = a.get("subtype", "unknown")
        acct_types[sub] = acct_types.get(sub, 0) + 1
    acct_str = ", ".join(f"{k}({v})" for k, v in acct_types.items())
    print(f"✅ {len(response.holdings)} positions  [{acct_str}]")

# Add manual liquid accounts
for name, balance in MANUAL_LIQUID_ACCOUNTS.items():
    all_cash_accounts.append({
        "brokerage": "manual",
        "account_name": name,
        "account_subtype": "checking",
        "category": "liquid",
        "balance": balance,
    })

if all_cash_accounts:
    print(f"\n💵 Cash accounts found:")
    for a in all_cash_accounts:
        print(f"   {a['brokerage']:12s} {a['account_name']:30s} ${a['balance']:>10,.2f}  ({a['account_subtype']})")
    total_cash = sum(a["balance"] for a in all_cash_accounts)
    print(f"   {'TOTAL':12s} {'':30s} ${total_cash:>10,.2f}")

print(f"\n📊 Total investment positions across all brokerages: {len(all_holdings)}")

Fetching holdings from 'Robinhood'... 

✅ 7 positions  [ira(1), brokerage(1)]
Fetching holdings from 'Sofi'... 

✅ 20 positions  [savings(1), brokerage(2), cash management(2), checking(1)]
Fetching holdings from 'Stash'... 

✅ 32 positions  [brokerage(1)]
Fetching holdings from 'Acorns'... 

✅ 4 positions  [roth(1), checking(1), brokerage(1)]
Fetching holdings from 'Wealthfront'... 

✅ 7 positions  [roth(1), checking(1)]

💵 Cash accounts found:
   Sofi         SoFi Savings                   $  1,911.45  (savings)
   Sofi         House down payment             $ 18,721.91  (cash management)
   Sofi         Parents fund                   $    354.02  (cash management)
   Sofi         SoFi Checking                  $    702.84  (checking)
   Acorns       Acorns Checking                $  1,163.00  (checking)
   Wealthfront  Individual Cash Account        $     10.79  (checking)
   manual       BofA Checking                  $  4,614.00  (checking)
   TOTAL                                       $ 27,478.01

📊 Total investment positions across all brokerages: 70


## Portfolio DataFrame

In [3]:
df = pd.DataFrame(all_holdings)

# Calculate gain/loss
df["gain_loss"] = df["value"] - df["cost_basis"]
df["gain_loss_pct"] = ((df["value"] - df["cost_basis"]) / df["cost_basis"] * 100).round(2)

# Sort by value descending
df = df.sort_values("value", ascending=False).reset_index(drop=True)

df

,brokerage,ticker,name,type,quantity,cost_basis,price,value,currency,account_id,account_name,account_subtype,account_type,gain_loss,gain_loss_pct
0,Stash,AOR,iShares Trust - iShares Core 60/40 Balanced Al...,etf,229.144770,12222.270290,65.620,15036.479807,USD,kVaO6gY9kXcDX86PMRzMsOB0LkYdK7U3K6yVr,Personal,brokerage,taxable,2814.209517,23.03
1,Robinhood,JEPQ,J.P. Morgan Exchange-Traded Fund Trust - JPMor...,etf,225.595501,11894.071599,56.840,12822.848277,USD,qNm3vyY7QNCqD1jD7AEKtwVexyPe8xUE89pow,Robinhood traditional IRA,ira,retirement,928.776678,7.81
2,Stash,SKYY,First Trust Exchange-Traded Fund III - First T...,etf,105.711020,9184.804640,114.120,12063.741602,USD,kVaO6gY9kXcDX86PMRzMsOB0LkYdK7U3K6yVr,Personal,brokerage,taxable,2878.936962,31.34
3,Robinhood,JEPI,J.P. Morgan Exchange-Traded Fund Trust - JPMor...,etf,180.087880,10612.956953,58.095,10462.205389,USD,BbMn8ZEvgbHn3wk30RzaHvaVk0bVJkt7wnEam,Robinhood individual,brokerage,taxable,-150.751564,-1.42
4,Robinhood,JEPQ,J.P. Morgan Exchange-Traded Fund Trust - JPMor...,etf,181.027230,10499.995703,56.840,10289.587753,USD,BbMn8ZEvgbHn3wk30RzaHvaVk0bVJkt7wnEam,Robinhood individual,brokerage,taxable,-210.407949,-2.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,Stash,GSK,GSK Plc - ADR,equity,1.725620,73.148160,54.560,94.149827,USD,kVaO6gY9kXcDX86PMRzMsOB0LkYdK7U3K6yVr,Personal,brokerage,taxable,21.001667,28.71
66,Sofi,CUR:USD,U S Dollar,cash,71.830000,NaN,1.000,71.830000,USD,AXODdQgXLmH44xwDkmvesJovoO7OqnHbQKZ0Z,SoFi Robo,brokerage,taxable,NaN,NaN
67,Sofi,CUR:USD,U S Dollar,cash,43.090000,NaN,1.000,43.090000,USD,kO1DdL7OJwFDD4r896dgIOBAB545PRT30QpB3,SoFi Self-directed,brokerage,taxable,NaN,NaN
68,Wealthfront,CUR:USD,U S Dollar,cash,19.530000,NaN,1.000,19.530000,USD,KqBqLkyvd7uy7YknwZp7I08Bj0Z3g3sQEoz1j,Roth IRA,roth,retirement,NaN,NaN


## Portfolio Summary

In [4]:
total_value = df["value"].sum()
total_cost = df["cost_basis"].sum()
total_gain = total_value - total_cost
gain_pct = (total_gain / total_cost * 100) if total_cost > 0 else 0

print("═" * 50)
print("         PORTFOLIO SUMMARY")
print("═" * 50)
print(f"  Total Value:       ${total_value:>12,.2f}")
print(f"  Total Cost Basis:  ${total_cost:>12,.2f}")
print(f"  Total Gain/Loss:   ${total_gain:>12,.2f}  ({gain_pct:+.2f}%)")
print(f"  Positions:         {len(df):>12}")
print(f"  Brokerages:        {df['brokerage'].nunique():>12}")
print("═" * 50)

══════════════════════════════════════════════════
         PORTFOLIO SUMMARY
══════════════════════════════════════════════════
  Total Value:       $  178,792.58
  Total Cost Basis:  $  112,346.93
  Total Gain/Loss:   $   66,445.66  (+59.14%)
  Positions:                   70
  Brokerages:                   5
══════════════════════════════════════════════════


## Breakdown by Brokerage

In [5]:
brokerage_summary = (
    df.groupby("brokerage")
    .agg(
        positions=("ticker", "count"),
        total_value=("value", "sum"),
        total_cost=("cost_basis", "sum"),
        total_gain=("gain_loss", "sum"),
    )
    .sort_values("total_value", ascending=False)
)

brokerage_summary["pct_of_portfolio"] = (
    (brokerage_summary["total_value"] / total_value * 100).round(1)
)

brokerage_summary

,positions,total_value,total_cost,total_gain,pct_of_portfolio
brokerage,,,,,
Stash,32,75744.297287,55019.118680,20723.828607,42.4
Robinhood,7,52234.604772,50020.758884,2213.845888,29.2
Sofi,20,31909.671565,0.000000,0.000000,17.8
Acorns,4,10311.260000,0.000000,0.000000,5.8
Wealthfront,7,8592.750000,7307.050000,1266.170000,4.8


## Breakdown by Asset Type

In [6]:
type_summary = (
    df.groupby("type")
    .agg(
        positions=("ticker", "count"),
        total_value=("value", "sum"),
    )
    .sort_values("total_value", ascending=False)
)

type_summary["pct_of_portfolio"] = (
    (type_summary["total_value"] / total_value * 100).round(1)
)

type_summary

,positions,total_value,pct_of_portfolio
type,,,
etf,47,142256.472217,79.6
equity,18,34898.661408,19.5
cash,5,1637.450000,0.9


## Top Holdings

In [7]:
# Show top 10 holdings by value (exclude cash)
top_holdings = (
    df[df["type"] != "cash"]
    .head(10)
    [["ticker", "name", "brokerage", "quantity", "price", "value", "gain_loss", "gain_loss_pct"]]
)

print("Top 10 Holdings by Value:")
top_holdings

Top 10 Holdings by Value:


,ticker,name,brokerage,quantity,price,value,gain_loss,gain_loss_pct
0,AOR,iShares Trust - iShares Core 60/40 Balanced Al...,Stash,229.144770,65.620,15036.479807,2814.209517,23.03
1,JEPQ,J.P. Morgan Exchange-Traded Fund Trust - JPMor...,Robinhood,225.595501,56.840,12822.848277,928.776678,7.81
2,SKYY,First Trust Exchange-Traded Fund III - First T...,Stash,105.711020,114.120,12063.741602,2878.936962,31.34
3,JEPI,J.P. Morgan Exchange-Traded Fund Trust - JPMor...,Robinhood,180.087880,58.095,10462.205389,-150.751564,-1.42
4,JEPQ,J.P. Morgan Exchange-Traded Fund Trust - JPMor...,Robinhood,181.027230,56.840,10289.587753,-210.407949,-2.00
5,JEPI,J.P. Morgan Exchange-Traded Fund Trust - JPMor...,Robinhood,175.440652,58.095,10192.224678,192.230322,1.92
6,VGT,Vanguard World Fund - Vanguard Information Tec...,Stash,13.807940,731.480,10100.231951,5100.550591,102.02
7,SUSA,BlackRock Institutional Trust Company N.A. - i...,Stash,45.848680,137.360,6297.774685,2297.774685,57.44
8,SOFI,SoFi Technologies Inc,Sofi,331.167270,18.900,6259.061403,NaN,NaN
9,MSFT,Microsoft Corporation,Stash,14.739250,412.400,6078.466700,1053.466700,20.96


## Export for Claude Analysis (Phase 3 Preview)

This saves the portfolio as a JSON file that we'll later feed into Claude's analysis prompt.

In [8]:
# Build account category summary from investment holdings
cat_summary = {}
for _, row in df.iterrows():
    cat = row.get("account_type", "other") or "other"
    if cat not in cat_summary:
        cat_summary[cat] = {"value": 0.0, "positions": 0}
    cat_summary[cat]["value"] += row.get("value", 0) or 0
    cat_summary[cat]["positions"] += 1

# Add cash account balances to liquid category
for cash_acct in all_cash_accounts:
    cat = cash_acct["category"]
    if cat not in cat_summary:
        cat_summary[cat] = {"value": 0.0, "positions": 0}
    cat_summary[cat]["value"] += cash_acct["balance"]
    cat_summary[cat]["positions"] += 1

for c in cat_summary:
    cat_summary[c]["value"] = round(cat_summary[c]["value"], 2)

portfolio_export = {
    "summary": {
        "total_value": round(total_value, 2),
        "total_cost_basis": round(total_cost, 2),
        "total_gain_loss": round(total_gain, 2),
        "total_positions": len(df),
        "brokerages": list(df["brokerage"].unique()),
        "account_categories": cat_summary,
    },
    "holdings": df.to_dict(orient="records"),
    "cash_accounts": all_cash_accounts,
}

export_path = os.path.join("..", "portfolio_snapshot.json")
with open(export_path, "w") as f:
    json.dump(portfolio_export, f, indent=2, default=str)

print(f"💾 Portfolio snapshot saved to {export_path}")
print(f"   This file will be fed to Claude for analysis in Phase 3.")
print(f"\n📂 Account categories:")
for cat, info in cat_summary.items():
    print(f"   {cat:12s}: ${info['value']:>12,.2f}  ({info['positions']} positions)")

💾 Portfolio snapshot saved to ../portfolio_snapshot.json
   This file will be fed to Claude for analysis in Phase 3.

📂 Account categories:
   taxable     : $  141,727.67  (60 positions)
   retirement  : $   37,064.91  (10 positions)
   liquid      : $   27,478.01  (7 positions)


In [ ]:
# Save raw Plaid holdings responses for audit/replay and Supabase storage
import os
import json
from datetime import datetime

RAW_RESPONSES_DIR = os.path.join("..", "raw_plaid_responses")
os.makedirs(RAW_RESPONSES_DIR, exist_ok=True)

# We re-fetch a lightweight summary here — the real raw responses were captured
# during the loop above. Here we store the full serialized holdings+securities
# payload per brokerage so sync_to_supabase.py can load them.
for nickname, token in access_tokens.items():
    try:
        from plaid.model.investments_holdings_get_request import InvestmentsHoldingsGetRequest as _HReq
        resp = client.investments_holdings_get(_HReq(access_token=token))
        raw_payload = {
            "brokerage": nickname,
            "endpoint": "investments/holdings/get",
            "fetched_at": datetime.utcnow().isoformat(),
            "accounts": [
                {
                    "account_id": a.account_id,
                    "name": a.name,
                    "type": str(a.type) if a.type else None,
                    "subtype": str(a.subtype) if a.subtype else None,
                }
                for a in resp.accounts
            ],
            "holdings": [
                {
                    "account_id": h.account_id,
                    "security_id": h.security_id,
                    "quantity": h.quantity,
                    "cost_basis": h.cost_basis,
                    "institution_price": h.institution_price,
                    "institution_value": h.institution_value,
                    "iso_currency_code": h.iso_currency_code,
                }
                for h in resp.holdings
            ],
            "securities": [
                {
                    "security_id": s.security_id,
                    "name": s.name,
                    "ticker_symbol": s.ticker_symbol,
                    "type": str(s.type) if s.type else None,
                    "close_price": s.close_price,
                    "iso_currency_code": s.iso_currency_code,
                }
                for s in resp.securities
            ],
        }
        ts = datetime.utcnow().strftime("%Y%m%dT%H%M%S")
        filename = f"holdings_{nickname}_{ts}.json"
        path = os.path.join(RAW_RESPONSES_DIR, filename)
        with open(path, "w") as f:
            json.dump(raw_payload, f, indent=2, default=str)
        print(f"   💾 Saved raw holdings response → raw_plaid_responses/{filename}")
    except Exception as e:
        print(f"   ⚠️  Could not save raw response for {nickname}: {e}")

print(f"\n✅ Raw holdings responses saved to {RAW_RESPONSES_DIR}")

## Save Portfolio History Snapshot

Appends a summary snapshot to `portfolio_history.json` for long-term growth tracking.
This builds up over time — each pipeline run adds one entry (deduplicated by date).

In [9]:
from datetime import datetime
import stat

HISTORY_FILE = os.path.join("..", "portfolio_history.json")

# Build today's snapshot
today_str = datetime.now().strftime("%Y-%m-%d")

# Brokerage breakdown (reuses brokerage_summary from cell 10)
brokerages_breakdown = {}
for name, row in brokerage_summary.iterrows():
    brokerages_breakdown[name] = {
        "value": round(row["total_value"], 2),
        "positions": int(row["positions"]),
    }

# Asset type breakdown (reuses type_summary from cell 12)
asset_breakdown = {}
for atype, row in type_summary.iterrows():
    asset_breakdown[atype] = {
        "value": round(row["total_value"], 2),
        "pct": float(row["pct_of_portfolio"]),
    }

snapshot = {
    "date": today_str,
    "total_value": round(total_value, 2),
    "total_cost_basis": round(total_cost, 2),
    "total_gain_loss": round(total_gain, 2),
    "gain_loss_pct": round(gain_pct, 2),
    "total_positions": len(df),
    "brokerages": brokerages_breakdown,
    "asset_types": asset_breakdown,
    "account_categories": cat_summary,  # NEW: taxable/retirement/liquid/other breakdown
}

# Load existing history or start fresh
if os.path.exists(HISTORY_FILE):
    with open(HISTORY_FILE, "r") as f:
        history = json.load(f)
else:
    history = []

# Deduplicate: replace existing entry for today, or append
existing_idx = next((i for i, h in enumerate(history) if h["date"] == today_str), None)
if existing_idx is not None:
    history[existing_idx] = snapshot
    action = "Updated"
else:
    history.append(snapshot)
    action = "Added"

# Save
with open(HISTORY_FILE, "w") as f:
    json.dump(history, f, indent=2, default=str)
os.chmod(HISTORY_FILE, stat.S_IRUSR | stat.S_IWUSR)

print(f"📈 {action} portfolio snapshot for {today_str}")
print(f"   History file: {HISTORY_FILE} ({len(history)} total entries)")
print(f"   Value: ${snapshot['total_value']:,.2f}  |  Gain: {snapshot['gain_loss_pct']:+.2f}%")

📈 Added portfolio snapshot for 2026-03-06
   History file: ../portfolio_history.json (11 total entries)
   Value: $178,792.58  |  Gain: +59.14%
